[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_03_main_transfer.ipynb)

# Module 6, Vision: Reusing What Another Model Already Learned

**Notebook:** `06_03_main_transfer`

## Where this fits

In `06_02`, our CNN started from random weights and learned visual features from only 3,500 EuroSAT training images.

That raises an obvious question:

> **Why learn vision from scratch if another network has already learned useful visual features from millions of images?**

That is the idea behind **transfer learning**.

We start with a pretrained vision model, remove its original classifier, and adapt the learned representation to our task.

The progression is now:

\[
\text{human-designed features}
\rightarrow
\text{features learned from our data}
\rightarrow
\text{features learned elsewhere and reused}
\]

We will use **MobileNetV2 pretrained on ImageNet**.

## Two stages

### Stage 1 — Feature extraction

Freeze the pretrained backbone.

Only a new EuroSAT classification head learns.

### Stage 2 — Fine-tuning

Unfreeze a small part of the backbone and update it with a much smaller learning rate.

The goal is not to relearn vision.

It is to gently adapt a useful pretrained representation to satellite imagery.

## The key question

> **How much can we gain by starting with a representation that already knows something about visual structure?**

## 0) Setup

This notebook runs independently in Colab.

It downloads:

- EuroSAT from Zenodo;
- MobileNetV2's pretrained ImageNet weights on first use.

A GPU is recommended for the training sections, but the model is intentionally lightweight.

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

import tensorflow as tf

SEED = 1955
N_SAMPLES = 5000
BATCH_SIZE = 64

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 1) Load the same EuroSAT sample

The sampling and splitting logic is the same as `06_01` and `06_02`:

- deterministic 5,000-image sample;
- 70% training;
- 10% validation;
- 20% test.

That keeps the comparison across approaches consistent even though each notebook runs independently.

In [ ]:
DATA_DIR = "assets/data"
ZIP_PATH = os.path.join(DATA_DIR, "EuroSAT_RGB.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "EuroSAT_RGB")
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading EuroSAT (~90 MB, one-time)...")
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

if not os.path.exists(EXTRACT_DIR):
    print("Extracting EuroSAT...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

print("EuroSAT ready.")

In [ ]:
EXPECTED_CLASSES = {
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
}

def find_class_dir(root):
    for current_root, dirs, _ in os.walk(root):
        if EXPECTED_CLASSES.issubset(set(dirs)):
            return current_root
    raise RuntimeError(f"Could not find EuroSAT class folders under {root}")

CLASS_DIR = find_class_dir(EXTRACT_DIR)
label_names = sorted(EXPECTED_CLASSES)
num_classes = len(label_names)

records = []

for class_id, class_name in enumerate(label_names):
    class_path = os.path.join(CLASS_DIR, class_name)

    for filename in sorted(os.listdir(class_path)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff")):
            records.append(
                (os.path.join(class_path, filename), class_id, class_name)
            )

records = sorted(records, key=lambda x: x[0])

paths = np.array([r[0] for r in records], dtype=object)
labels = np.array([r[1] for r in records], dtype=np.int64)

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(
    len(paths),
    size=N_SAMPLES,
    replace=False,
)

sample_paths = paths[sample_idx]
sample_labels = labels[sample_idx]

idx_all = np.arange(N_SAMPLES)

idx_dev, idx_test = train_test_split(
    idx_all,
    test_size=0.20,
    stratify=sample_labels,
    random_state=SEED,
)

idx_train, idx_val = train_test_split(
    idx_dev,
    test_size=0.125,
    stratify=sample_labels[idx_dev],
    random_state=SEED,
)

print(
    f"train={len(idx_train):,}  "
    f"validation={len(idx_val):,}  "
    f"test={len(idx_test):,}"
)
print("classes:", label_names)

In [ ]:
X_img = np.empty(
    (N_SAMPLES, 64, 64, 3),
    dtype=np.uint8,
)

for i, path in enumerate(sample_paths):
    with Image.open(path) as image:
        X_img[i] = np.asarray(
            image.convert("RGB"),
            dtype=np.uint8,
        )

y = sample_labels

X_train = X_img[idx_train]
X_val = X_img[idx_val]
X_test = X_img[idx_test]

y_train = y[idx_train]
y_val = y[idx_val]
y_test = y[idx_test]

print("train:", X_train.shape)
print("validation:", X_val.shape)
print("test:", X_test.shape)

### Teaching note about the test set

As in the earlier notebooks, we inspect test performance throughout the module to make the comparison visible.

In a formal model-development workflow, repeatedly looking at test performance while changing the model would make the test set part of the development process. A separate final holdout would normally be retained.

## 2) Keep the data pipeline simple

The TensorFlow datasets will supply the original `64 × 64` RGB images as floating-point tensors.

The model itself will handle:

1. training-time augmentation;
2. resizing;
3. MobileNetV2-specific preprocessing.

Keeping those transformations inside the model makes the preprocessing pipeline part of the saved model definition.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(images, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )

    ds = ds.map(
        lambda x, y: (
            tf.cast(x, tf.float32),
            y,
        ),
        num_parallel_calls=AUTOTUNE,
    )

    if shuffle:
        ds = ds.shuffle(
            2048,
            seed=SEED,
        )

    return (
        ds
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

ds_train = make_dataset(
    X_train,
    y_train,
    shuffle=True,
)

ds_val = make_dataset(
    X_val,
    y_val,
)

ds_test = make_dataset(
    X_test,
    y_test,
)

for images, labels_batch in ds_train.take(1):
    print("image batch:", images.shape, images.dtype)
    print(
        "pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images)),
    )

## 3) Resize for the pretrained backbone

EuroSAT images are natively `64 × 64`.

We will resize them to `96 × 96` before passing them into MobileNetV2.

A critical point:

> **Resizing does not create new image detail.**

The original information is still only `64 × 64`.

The resize simply presents that information on a larger grid, which can be convenient for a pretrained architecture and gives later convolutional layers more spatial positions to work with.

MobileNetV2 with the original ImageNet classification head expects `224 × 224`, but when `include_top=False`, Keras allows smaller spatial inputs.

We also use MobileNetV2's required preprocessing, which maps pixel values into the range expected by the pretrained weights.

## 4) Stage 1: use the pretrained network as a feature extractor

A pretrained CNN has already learned a hierarchy of visual features.

We remove the original ImageNet classifier with:

```python
include_top=False
```

Then we freeze the backbone:

```python
backbone.trainable = False
```

During Stage 1:

- MobileNetV2 provides a fixed visual representation;
- only our new EuroSAT classification head learns.

Conceptually:

\[
\text{EuroSAT image}
\rightarrow
\text{pretrained visual features}
\rightarrow
\text{new 10-class head}
\]

This is transfer learning in its simplest form.

In [ ]:
INPUT_SHAPE = (64, 64, 3)
BACKBONE_SIZE = (96, 96)

backbone = tf.keras.applications.MobileNetV2(
    input_shape=BACKBONE_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)

backbone.trainable = False

inputs = tf.keras.Input(
    shape=INPUT_SHAPE,
)

x = tf.keras.layers.RandomFlip(
    "horizontal_and_vertical",
    seed=SEED,
)(inputs)

x = tf.keras.layers.Resizing(
    *BACKBONE_SIZE,
)(x)

x = tf.keras.applications.mobilenet_v2.preprocess_input(
    x
)

# training=False is intentional.
# It keeps MobileNetV2's BatchNorm layers in inference mode,
# which is especially important when we later fine-tune.
x = backbone(
    x,
    training=False,
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(
    0.20,
    seed=SEED,
)(x)

outputs = tf.keras.layers.Dense(
    num_classes,
    activation="softmax",
)(x)

transfer_model = tf.keras.Model(
    inputs,
    outputs,
    name="mobilenetv2_transfer",
)

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

transfer_model.summary()

In [ ]:
def count_parameters(model):
    trainable = sum(
        int(np.prod(v.shape))
        for v in model.trainable_variables
    )

    total = model.count_params()

    return trainable, total


trainable, total = count_parameters(
    transfer_model
)

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters    : {total:,}")
print(
    f"Percent trainable   : "
    f"{100 * trainable / total:.2f}%"
)

### What is actually learning?

At this stage, almost all parameters are frozen.

We are not retraining MobileNetV2.

We are learning a relatively small mapping from:

> **pretrained visual representation → EuroSAT class**

This is why transfer learning can work well with limited task-specific data.

In [ ]:
def make_callbacks(
    patience_lr=3,
    patience_stop=7,
):
    return [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience_stop,
            restore_best_weights=True,
            verbose=1,
        ),
    ]


history_stage1 = transfer_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=30,
    callbacks=make_callbacks(),
    verbose=2,
)

stage1_loss, stage1_acc = transfer_model.evaluate(
    ds_test,
    verbose=0,
)

print(
    f"Stage 1 test accuracy: "
    f"{stage1_acc:.3f}"
)

## 5) Stage 2: fine-tune part of the representation

The ImageNet representation is useful, but EuroSAT is a different domain.

Natural photographs and satellite images do not have identical visual statistics.

Fine-tuning lets some of the deeper pretrained filters adapt.

We will:

1. unfreeze the backbone;
2. keep most early layers frozen;
3. keep BatchNormalization layers frozen;
4. allow only the top portion of the backbone to update;
5. recompile with a learning rate **100× smaller**.

Why so cautious?

Because the starting representation is valuable.

Large updates can destroy useful pretrained weights faster than our small dataset can improve them.

### Why special care with BatchNormalization?

MobileNetV2 contains many BatchNormalization layers.

These layers maintain moving statistics about their inputs.

When fine-tuning a pretrained network on a small new dataset, abruptly updating those statistics can damage the pretrained representation.

Keras therefore recommends running the pretrained backbone with:

```python
training=False
```

during fine-tuning.

That is why the model was built that way from the beginning.

In [ ]:
# Unfreeze the backbone, then selectively freeze most layers.
backbone.trainable = True

FINE_TUNE_LAST = 30
fine_tune_start = len(backbone.layers) - FINE_TUNE_LAST

for layer in backbone.layers[:fine_tune_start]:
    layer.trainable = False

# Keep BatchNormalization layers frozen.
for layer in backbone.layers[fine_tune_start:]:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization,
    ):
        layer.trainable = False

# Recompile whenever trainable status changes.
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable, total = count_parameters(
    transfer_model
)

print(f"Backbone layers: {len(backbone.layers)}")
print(f"Fine-tuning from layer: {fine_tune_start}")
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters    : {total:,}")
print(
    f"Percent trainable   : "
    f"{100 * trainable / total:.2f}%"
)

In [ ]:
history_stage2 = transfer_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=make_callbacks(
        patience_lr=3,
        patience_stop=6,
    ),
    verbose=2,
)

stage2_loss, stage2_acc = transfer_model.evaluate(
    ds_test,
    verbose=0,
)

print(
    f"Stage 2 test accuracy: "
    f"{stage2_acc:.3f}"
)

## 6) Visualize the two training stages

Stage 1 asks:

> Can ImageNet features already separate our EuroSAT classes?

Stage 2 asks:

> Can small updates make those features more useful for satellite imagery?

Fine-tuning does not have to produce a dramatic gain to be worthwhile.

Sometimes the frozen representation is already strong.

Sometimes the new domain benefits from modest adaptation.

The empirical question is whether validation performance improves without destabilizing the model.

In [ ]:
stage1_train = history_stage1.history["accuracy"]
stage1_val = history_stage1.history["val_accuracy"]

stage2_train = history_stage2.history["accuracy"]
stage2_val = history_stage2.history["val_accuracy"]

all_train = stage1_train + stage2_train
all_val = stage1_val + stage2_val

boundary = len(stage1_train)

plt.figure(
    figsize=(10, 4)
)

plt.plot(
    all_train,
    label="train",
)

plt.plot(
    all_val,
    label="validation",
)

plt.axvline(
    boundary - 0.5,
    linestyle="--",
    alpha=0.7,
    label="begin fine-tuning",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer learning: frozen features → fine-tuning")
plt.ylim(0, 1)
plt.legend()

plt.tight_layout()
plt.show()

## 7) Compare the two transfer-learning stages

The comparison we can make directly in this notebook is:

- frozen pretrained representation;
- partially fine-tuned pretrained representation.

When you run the full module, add the observed results from:

- `06_01`: random forest on 34 human-designed features;
- `06_02`: CNN learned from scratch.

Avoid assuming the progression must always be monotonic.

A more sophisticated method can perform worse if:

- optimization goes poorly;
- the pretraining domain transfers badly;
- the new dataset is very different;
- hyperparameters are inappropriate.

The method gives us an advantage, not a guarantee.

In [ ]:
results = pd.DataFrame(
    {
        "approach": [
            "Random chance",
            "Transfer: frozen backbone",
            "Transfer: partial fine-tuning",
        ],
        "test_accuracy": [
            1 / num_classes,
            stage1_acc,
            stage2_acc,
        ],
    }
)

results

## 8) Inspect the remaining errors

Transfer learning may improve accuracy substantially, but we still need to understand the mistakes.

We will inspect:

- per-class performance;
- the confusion matrix;
- confident errors.

Do not assume that every remaining error is caused by insufficient model capacity.

Some images may genuinely resemble more than one land-cover category at `64 × 64` resolution.

In [ ]:
probabilities = transfer_model.predict(
    ds_test,
    verbose=0,
)

predictions = probabilities.argmax(
    axis=1
)

print(
    classification_report(
        y_test,
        predictions,
        target_names=label_names,
        digits=3,
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    predictions,
)

fig, ax = plt.subplots(
    figsize=(9, 8)
)

im = ax.imshow(
    cm,
    cmap="Blues",
)

ax.set_xticks(range(num_classes))
ax.set_xticklabels(
    label_names,
    rotation=90,
    fontsize=9,
)

ax.set_yticks(range(num_classes))
ax.set_yticklabels(
    label_names,
    fontsize=9,
)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Fine-tuned MobileNetV2 confusion matrix")

for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            ),
        )

plt.colorbar(
    im,
    ax=ax,
    fraction=0.046,
)

plt.tight_layout()
plt.show()

In [ ]:
confidence = probabilities.max(axis=1)

wrong = np.where(
    predictions != y_test
)[0]

wrong = wrong[
    np.argsort(
        -confidence[wrong]
    )
]

n_show = min(
    8,
    len(wrong),
)

selected = wrong[:n_show]

fig, axs = plt.subplots(
    2,
    4,
    figsize=(12, 6),
)

for ax in axs.flat:
    ax.axis("off")

for test_pos, ax in zip(
    selected,
    axs.flat,
):
    ax.imshow(
        X_test[test_pos]
    )

    ax.set_title(
        f"true: {label_names[y_test[test_pos]]}\n"
        f"pred: {label_names[predictions[test_pos]]} "
        f"({confidence[test_pos]:.2f})",
        fontsize=9,
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## 9) The transfer-learning idea is bigger than MobileNet

MobileNetV2 still follows the traditional supervised-learning pattern:

1. use a pretrained image encoder;
2. add a new classifier;
3. train on labeled examples for the new task.

But notice what changed across these notebooks:

### Classical vision

\[
\text{humans define the representation}
\]

### CNN from scratch

\[
\text{the model learns a representation from our labeled task}
\]

### Transfer learning

\[
\text{the model reuses a representation learned from a much larger task}
\]

That final idea—**learn a broadly useful representation once, then reuse it**—is one of the foundations of modern AI.

## 10) Bridge to vision foundation models

Modern vision systems push transfer learning further.

A model such as **CLIP** learns from paired images and language.

Instead of learning only:

\[
\text{image}
\rightarrow
\text{class label}
\]

it learns compatible representations for both:

\[
\text{image}
\rightarrow
\text{image vector}
\]

and

\[
\text{text}
\rightarrow
\text{text vector}
\]

Now an image can be compared directly with text descriptions.

For a satellite tile, imagine candidate descriptions such as:

```text
"a satellite image of a forest"
"a satellite image of a highway"
"a satellite image of an industrial area"
"a satellite image of residential buildings"
```

A vision-language model can embed the image and each sentence into a shared space, then choose the closest description.

### Why this is a conceptual jump

With MobileNetV2, we still trained a new 10-class classifier.

With a vision-language foundation model, we can potentially classify using **language itself as the interface**.

That enables:

- zero-shot classification;
- image-text search;
- flexible categories defined at inference time;
- multimodal retrieval;
- image understanding as part of larger AI systems.

We will return to this idea when we move from task-specific models into **foundation models and multimodal AI**.

## Takeaways

### 1. Pretraining changes the starting point

The CNN no longer begins with random visual filters.

It begins with a representation learned from a much larger dataset.

### 2. Feature extraction and fine-tuning are different strategies

**Feature extraction:** freeze the representation and learn only a new head.

**Fine-tuning:** cautiously adapt part of the representation.

### 3. Fine-tuning should be conservative

Use:

- fewer trainable layers;
- a much smaller learning rate;
- careful treatment of BatchNormalization;
- validation-based stopping.

### 4. Resizing is not new information

Changing `64 × 64` to `96 × 96` changes the computational grid, not the underlying visual resolution.

### 5. Transfer learning points toward foundation models

The larger idea is:

> **Learn a useful representation once, then reuse it across tasks.**

Vision-language foundation models extend that idea by connecting visual representations with language.

---

## Where the module goes next

At this point we have moved through four increasingly automated representations:

\[
\text{pixels and hand filters}
\rightarrow
\text{human-engineered features}
\rightarrow
\text{task-learned CNN features}
\rightarrow
\text{pretrained reusable features}
\]

The student exercise will shift the question from:

> **Can we build a classifier?**

to:

> **Which vision system should we trust, and how do we know?**